<img src="https://udemedellin.edu.co/wp-content/uploads/2022/10/logo_udemedellin2.png" width="30%">

<b>ESPECIALIZACIÓN EN CIENCIA DE DATOS E INGELIGENCIA ARTIFICIAL</b>

<strong>Fundamentos de Estadística para Ciencia de Datos</strong>

# Sesión 05 — Taller práctico: estadística descriptiva (caso de estudio)

## Objetivos de aprendizaje

Al finalizar este taller serás capaz de:

- Aplicar el flujo completo de estadística descriptiva (tipos de datos, calidad de datos, descriptiva cualitativa y cuantitativa) sobre un caso real, integrando lo visto en las sesiones 1 a 4.
- Tomar y justificar decisiones de limpieza de datos (faltantes) según el contexto de cada variable, incluyendo cuándo la ausencia de un dato puede ser informativa y no aleatoria.
- Construir variables derivadas (*feature engineering*) cuando aportan una señal que las variables originales no dan por separado.
- Detectar y tratar atípicos (no solo detectarlos): comparar recorte (*capping*) contra transformación logarítmica según el objetivo del análisis.
- Describir variables cualitativas y cuantitativas, y relacionarlas con una variable target de negocio, sin extrapolar más allá de lo que los datos permiten.

In [3]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", palette="deep")
%matplotlib inline

pd.set_option("display.precision", 3)

## 1. Planteamiento del caso y carga de datos

**Contexto:** 

Eres analista de riesgo en una entidad financiera. El equipo de crédito quiere entender qué caracteriza a las solicitudes de préstamo aprobadas frente a las rechazadas, antes de construir cualquier modelo de aprobación.

**Objetivo del taller:** 

Revisar el dataset `loan` para explorarlo, limpiarlo y describirlo, identificando qué caracteriza a las solicitudes según su estado de aprobación, como insumo para decisiones posteriores. No se construye ningún modelo ni prueba de hipótesis en este taller.

**Fuente:** 

Dataset "Loan Prediction Practice Problem III", una competencia práctica de **Analytics Vidhya**. Cargamos el dataset completo (614 solicitudes en train, 367 en test) desde un mirror en GitHub con acceso directo por URL.

In [5]:
loan = pd.read_csv("https://raw.githubusercontent.com/sahutkarsh/loan-prediction-analytics-vidhya/master/train.csv")
loan_test = pd.read_csv("https://raw.githubusercontent.com/sahutkarsh/loan-prediction-analytics-vidhya/master/test.csv")
print(f"Filas: {loan.shape[0]}")
print(f"Columnas: {loan.shape[1]}")
loan.head()

Filas: 614
Columnas: 13


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


### Diccionario de datos

Este es el diccionario del dataset original "Loan Prediction Practice Problem III" (Analytics Vidhya).

| Variable | Significado |
|---|---|
| `Loan_ID` | ID único de la solicitud |
| `Gender` | Género del solicitante |
| `Married` | Estado civil del solicitante (casado/no) |
| `Dependents` | Número de dependientes económicos |
| `Education` | Nivel educativo (Graduate / Not Graduate) |
| `Self_Employed` | Si el solicitante es independiente |
| `ApplicantIncome` | Ingreso mensual del solicitante |
| `CoapplicantIncome` | Ingreso mensual del coaplicante (0 si no hay) |
| `LoanAmount` | Monto del préstamo solicitado, en miles |
| `Loan_Amount_Term` | Plazo del préstamo, en meses |
| `Credit_History` | Si el historial crediticio cumple los criterios del banco (1) o no (0) |
| `Property_Area` | Zona del inmueble (Urban / Semiurban / Rural) |
| `Loan_Status` | Si el préstamo fue aprobado (Y/N, aquí 1/0) — **variable target** |

### Clasificación de variables

Antes de describir nada, clasificamos cada columna según la taxonomía de la sesión 1:

| Variable | Tipo | Subtipo |
|---|---|---|
| `Loan_ID` | Identificador | No es una variable analítica, es una clave única por solicitud |
| `Gender`, `Married`, `Education`, `Self_Employed`, `Property_Area` | Cualitativa | Nominal |
| `Dependents` | Cualitativa | Ordinal — en el fondo es un conteo, pero el valor `"3+"` agrupa "3 o más" en una sola categoría abierta, así que se trata como ordinal en vez de forzarla a numérica |
| `ApplicantIncome`, `CoapplicantIncome`, `LoanAmount` | Cuantitativa | Continua (razón) |
| `Loan_Amount_Term` | Cuantitativa | Discreta (plazo en meses, toma pocos valores repetidos) |
| `Credit_History` | Cuantitativa (codificada 0/1) | En la práctica funciona como cualitativa binaria: 1 = tiene historial crediticio que cumple los criterios del banco, 0 = no lo cumple |
| `Loan_Status` | Cualitativa | Nominal binaria — **variable target de este taller** (1 = aprobado, 0 = rechazado) |

`Loan_ID` no aporta información analítica (es una clave única por fila, cardinalidad = número de filas), así que se descarta antes de cualquier análisis.